# 05 Web Tools and Browser Automation Strategy (OpenClaw, 2026)

## What This Lesson Is
Choose between web search/fetch and browser automation based on task constraints and risk profile.

## Scientific Lens
- Concept: Tool-selection policy minimizes risk while preserving task success.
- Measure: Correct tool selection rate on labeled web tasks.
- Validity Limit: Static policy may underperform on ambiguous, changing websites.


## How It Works
1. Label tasks by dynamic content/auth requirements.
2. Select minimal tool capability needed by policy.
3. Run live OpenClaw prompt requesting tool strategy and compare with deterministic baseline.


In [ ]:
import os
from openai import OpenAI

OPENCLAW_BASE_URL = os.getenv("OPENCLAW_BASE_URL", "http://127.0.0.1:18789").rstrip("/")
OPENCLAW_TOKEN = os.getenv("OPENCLAW_GATEWAY_TOKEN") or os.getenv("OPENAI_API_KEY") or ""
OPENCLAW_TOKEN_SOURCE = (
    "OPENCLAW_GATEWAY_TOKEN" if os.getenv("OPENCLAW_GATEWAY_TOKEN")
    else ("OPENAI_API_KEY" if os.getenv("OPENAI_API_KEY") else "<missing>")
)
OPENCLAW_AGENT_ID = os.getenv("OPENCLAW_AGENT_ID", "main")

print("OPENCLAW_BASE_URL:", OPENCLAW_BASE_URL)
print("OPENCLAW_TOKEN source:", OPENCLAW_TOKEN_SOURCE)
print("OPENCLAW_AGENT_ID:", OPENCLAW_AGENT_ID)


def build_client() -> OpenAI:
    # OpenClaw exposes an OpenAI-compatible Chat Completions endpoint at /v1/chat/completions.
    # Docs: https://docs.openclaw.ai/gateway/openai-http-api
    return OpenAI(base_url=f"{OPENCLAW_BASE_URL}/v1", api_key=OPENCLAW_TOKEN or "local-dev-token")


def ask_openclaw(prompt: str, user: str = "lesson-user", temperature: float = 0.2) -> str:
    client = build_client()
    resp = client.chat.completions.create(
        model="openclaw",
        messages=[{"role": "user", "content": prompt}],
        user=user,
        temperature=temperature,
        extra_headers={"x-openclaw-agent-id": OPENCLAW_AGENT_ID},
    )
    return resp.choices[0].message.content or ""


## Code Walkthrough
- `Deterministic Demo` defines and validates the decision logic.
- `Live Demo` executes a real OpenClaw agent call through the OpenAI-compatible gateway API.


In [ ]:
# Deterministic Demo
tasks = [
    {"name":"public docs lookup","needs_login":False,"js_heavy":False},
    {"name":"submit secured form","needs_login":True,"js_heavy":True},
]
def choose(t):
    return "browser" if (t["needs_login"] or t["js_heavy"]) else "web_fetch"
assert choose(tasks[0]) == "web_fetch"
assert choose(tasks[1]) == "browser"


In [ ]:
# Live Demo
try:
    q = "For an authenticated JS-heavy workflow, should the agent use web_fetch or browser? Explain why."
    print(ask_openclaw(q, user="tool-policy"))
except Exception as exc:
    print(f"Live demo call failed: {exc}")
    print("Set OPENCLAW_GATEWAY_TOKEN in .env (or export OPENAI_API_KEY) and rerun.")


## Applied Labs
1. Add a denylist policy where browser is disallowed for untrusted sources.
2. Measure latency delta between browser and fetch strategies over 20 tasks.
3. Create a fallback rule when browser tool is denied by policy.

## Validation Checklist
- Tool policy is explicit and grounded in constraints.
- Security/risk tradeoffs are encoded in decision logic.
- Live call tests strategy reasoning via OpenClaw gateway integration.

## Further Reading
- OpenClaw web tools: https://docs.openclaw.ai/tools/web
- OpenClaw browser tool: https://docs.openclaw.ai/tools/browser
